# Kartoffelmachine · Prueba del modelo de clasificación (TRL 3)

Ejecuta las celdas de arriba hacia abajo. Este cuaderno obtiene las predicciones **reales** del modelo NCNN proporcionado y guarda un CSV de resultados. No calcula una precisión definitiva: cuatro imágenes no constituyen un conjunto de validación suficiente.

In [ ]:
%pip -q install ultralytics ncnn pandas pillow

## 1. Subir el ZIP de prueba
Carga `TRL3_Paso11_Datos_IA.zip` cuando aparezca el selector de archivos.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile
uploaded = files.upload()
zip_path = next((Path(name) for name in uploaded if name.endswith('.zip')), None)
assert zip_path is not None, 'Debes subir el archivo ZIP proporcionado.'
with zipfile.ZipFile(zip_path) as z:
    z.extractall('/content/trl3_ia')
print('Datos extraídos correctamente')

## 2. Ejecutar las predicciones
El programa carga los pesos NCNN y compara el resultado con las etiquetas asignadas a las fotografías.

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import pandas as pd
import numpy as np

base = Path('/content/trl3_ia')
model_path = base / 'best_ncnn_model'
assert (model_path / 'model.ncnn.param').exists(), 'Falta model.ncnn.param'
assert (model_path / 'model.ncnn.bin').exists(), 'Falta model.ncnn.bin'
model = YOLO(str(model_path), task='classify')
expected = {'Papa_BE_4.jpg': 'Buen estado', 'Papa_BE_13.jpg': 'Buen estado', 'Papa_D_7.jpg': 'Defectuoso', 'Papa_D_5.jpg': 'Defectuoso'}
records=[]
for filename,actual in expected.items():
    result = model.predict(source=str(base / 'Imagenes_prueba' / filename), imgsz=224, device='cpu', verbose=False)[0]
    labels = result.names
    scores = result.probs.data.cpu().numpy().astype(float)
    top = int(np.argmax(scores))
    predicted = labels[top]
    row = {'imagen': filename, 'etiqueta_referencia': actual, 'prediccion_modelo': predicted, 'confianza': round(float(scores[top]),6), 'coincide': predicted==actual}
    for i,score in enumerate(scores): row[f'prob_{labels[i]}'] = round(float(score),6)
    records.append(row)
resultados=pd.DataFrame(records)
display(resultados)
resultados.to_csv('/content/TRL3_Predicciones_IA.csv',index=False)
print(f'Coincidencias en estas cuatro fotos: {sum(resultados.coincide)}/{len(resultados)} (NO es la precisión de validación del modelo)')

## 3. Revisar las fotografías y descargar los resultados
Las imágenes originales no se modifican.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
fig,axs=plt.subplots(1,4,figsize=(15,4))
for ax,(filename,ref) in zip(axs,expected.items()):
    r=next(x for x in records if x['imagen']==filename)
    ax.imshow(Image.open(base/'Imagenes_prueba'/filename))
    ax.axis('off')
    ax.set_title(f"{filename}\nRef: {ref}\nPred: {r['prediccion_modelo']} ({r['confianza']:.1%})", fontsize=9)
plt.tight_layout()
fig.savefig('/content/TRL3_Comparacion_IA.png',dpi=180,bbox_inches='tight')
plt.show()
from google.colab import files
files.download('/content/TRL3_Predicciones_IA.csv')
files.download('/content/TRL3_Comparacion_IA.png')